# Precomputed vs. On-Demand BoxMap

CMGDB's `BoxMap` supports two evaluation paths for the map enclosure required by
the adaptive subdivision algorithm:

- **On-demand** (`mode='corners'`, default): the callable `F(rect)` is invoked
  once per phase-space box, evaluating the map at the $2^d$ corners and taking
  the componentwise min/max as the enclosing rectangle.
- **Precomputed**: `CMGDB.make_precomputed_box_map` evaluates the map on the
  finest corner lattice in a single batched pass before the CMGDB subdivision
  loop begins.  Subsequent `F(rect)` calls are $O(1)$ table lookups.

Both paths must produce **identical** Morse decompositions.  The precomputed
path eliminates per-call Python overhead during the subdivision loop — a
significant gain when `F` is expensive (e.g., a trained neural network) and the
CMGDB inner loop calls `F` millions of times.  For a cheap analytic map the
on-demand path is faster overall because the precomputed overhead dominates.

This notebook demonstrates both paths on the two-dimensional Leslie population
model with paper parameters $\bar\theta = (23.5, 23.5)$, asserts their Morse
outputs are identical, and reports the wall-clock split.

The Leslie (Beverton–Holt) map is
$$
F\begin{pmatrix}x_0\\x_1\end{pmatrix}
= \begin{pmatrix}
(\theta_1 x_0 + \theta_2 x_1)\, e^{-0.1(x_0+x_1)} \\
s\, x_0
\end{pmatrix},
\qquad s = 0.7,\quad \theta_1 = \theta_2 = 23.5.
$$

In [1]:
!pip install git+https://github.com/bernardorivas/CMGDB.git

zsh:1: command not found: pip


In [2]:
import math
import time

import CMGDB

## Leslie map definition

The box-map enclosure evaluates $F$ at all $2^2 = 4$ corners of a rectangle and
returns the componentwise bounding box of the four images.  No outer padding is
applied here; the bounds already enclose the attractor with margin.

In [3]:
THETA_1 = 23.5
THETA_2 = 23.5
SURVIVAL = 0.7


def leslie_map(x):
    """Point map for the 2D Leslie model."""
    return [
        (THETA_1 * x[0] + THETA_2 * x[1]) * math.exp(-0.1 * (x[0] + x[1])),
        SURVIVAL * x[0],
    ]


def leslie_box_map(rect):
    """Corner-based box-map enclosure (no padding)."""
    x0_lo, x1_lo, x0_hi, x1_hi = rect
    corners = [
        leslie_map([x0_lo, x1_lo]),
        leslie_map([x0_hi, x1_lo]),
        leslie_map([x0_lo, x1_hi]),
        leslie_map([x0_hi, x1_hi]),
    ]
    y0_vals = [c[0] for c in corners]
    y1_vals = [c[1] for c in corners]
    return [min(y0_vals), min(y1_vals), max(y0_vals), max(y1_vals)]

## CMGDB parameters

The phase-space domain $[0, 110] \\times [0, 77]$ encloses the attractor with
margin; the small negative offset on the lower bound follows the convention in
the CMGDB Leslie examples.  `subdiv_init=20, subdiv_max=22` gives a moderate
grid (around 400 000 boxes at the finest level) so each run completes in well
under a minute on a laptop.

In [4]:
SUBDIV_INIT = 20
SUBDIV_MIN  = 20
SUBDIV_MAX  = 22
SUBDIV_LIMIT = 10_000

LOWER_BOUNDS = [-0.001, -0.001]
UPPER_BOUNDS = [110.0,   77.0]

## On-demand BoxMap

The CMGDB `Model` is given the `leslie_box_map` callable directly.  The C++
subdivision loop calls `leslie_box_map(rect)` once per box during the
decomposition — millions of Python invocations at fine subdivisions.

In [5]:
model_on_demand = CMGDB.Model(
    SUBDIV_MIN, SUBDIV_MAX, SUBDIV_INIT, SUBDIV_LIMIT,
    LOWER_BOUNDS, UPPER_BOUNDS,
    leslie_box_map,
)

t0 = time.perf_counter()
morse_graph_on_demand, map_graph_on_demand = CMGDB.ComputeMorseGraph(model_on_demand)
t_on_demand = time.perf_counter() - t0

print(f"On-demand wall-clock:  {t_on_demand:.2f} s")
print(f"Morse graph vertices:  {morse_graph_on_demand.num_vertices()}")

On-demand wall-clock:  6.98 s
Morse graph vertices:  4


## Precomputed BoxMap

`CMGDB.make_precomputed_box_map` evaluates the map on the finest corner lattice
in a single batched pass before the CMGDB subdivision loop begins.  The
`mode='adaptive'` variant mirrors the adaptive subdivision grid used internally
by CMGDB: it stores one table entry per grid corner at depth `subdiv_max`, where
corner index $k$ along axis $i$ corresponds to the boundary of the $k$-th cell
at that axis's resolution.  Subsequent `box_map(rect)` calls are $O(1)$ table
lookups and return the same enclosure as a direct corner evaluation.

For an analytic map with cheap per-call cost (such as this Leslie map), the
precomputed path does not reduce total wall-clock time — the C++ subdivision work
dominates in both cases.  The benefit is significant only when `F` is expensive
to evaluate (e.g., a trained neural network requiring a PyTorch forward pass),
where per-call Python overhead multiplied over millions of box evaluations is the
bottleneck.

In [6]:
import numpy as np


def leslie_map_batched(points):
    """Vectorised Leslie map for make_precomputed_box_map.

    Parameters
    ----------
    points : ndarray of shape (n, 2)

    Returns
    -------
    ndarray of shape (n, 2)
    """
    points = np.asarray(points, dtype=np.float64)
    x0 = points[:, 0]
    x1 = points[:, 1]
    decay = np.exp(-0.1 * (x0 + x1))
    y0 = (THETA_1 * x0 + THETA_2 * x1) * decay
    y1 = SURVIVAL * x0
    return np.column_stack([y0, y1])


t0 = time.perf_counter()
precomputed_box_map = CMGDB.make_precomputed_box_map(
    leslie_map_batched,
    lower_bounds=LOWER_BOUNDS,
    upper_bounds=UPPER_BOUNDS,
    subdiv_max=SUBDIV_MAX,
    mode="adaptive",
    padding=False,
)
t_precompute = time.perf_counter() - t0

model_precomputed = CMGDB.Model(
    SUBDIV_MIN, SUBDIV_MAX, SUBDIV_INIT, SUBDIV_LIMIT,
    LOWER_BOUNDS, UPPER_BOUNDS,
    precomputed_box_map,
)

t0 = time.perf_counter()
morse_graph_precomputed, map_graph_precomputed = CMGDB.ComputeMorseGraph(model_precomputed)
t_cmgdb_precomputed = time.perf_counter() - t0
t_precomputed_total = t_precompute + t_cmgdb_precomputed

print(f"Precompute pass:       {t_precompute:.2f} s")
print(f"CMGDB (precomputed):   {t_cmgdb_precomputed:.2f} s")
print(f"Total wall-clock:      {t_precomputed_total:.2f} s")
print(f"Morse graph vertices:  {morse_graph_precomputed.num_vertices()}")

Precompute pass:       1.90 s
CMGDB (precomputed):   41.69 s
Total wall-clock:      43.59 s
Morse graph vertices:  4


## Correctness assertion

Both paths must produce the same Morse decomposition: equal vertex count, equal
adjacency structure (edge set), and equal Conley-index annotations per vertex.
The precomputed lookup replaces the per-call Python overhead but does not alter
what the box map computes: each table lookup returns the same result as calling
`leslie_box_map` at the corresponding grid rectangle.

In [7]:
n_od = morse_graph_on_demand.num_vertices()
n_pc = morse_graph_precomputed.num_vertices()

assert n_od == n_pc, (
    f"Vertex count mismatch: on-demand={n_od}, precomputed={n_pc}"
)

# Edge sets
edges_od = {
    (u, v)
    for u in range(n_od)
    for v in morse_graph_on_demand.adjacencies(u)
}
edges_pc = {
    (u, v)
    for u in range(n_pc)
    for v in morse_graph_precomputed.adjacencies(u)
}
assert edges_od == edges_pc, (
    f"Edge set mismatch: on-demand={sorted(edges_od)}, "
    f"precomputed={sorted(edges_pc)}"
)

# Conley-index annotations
annotations_od = [
    tuple(morse_graph_on_demand.annotations(v)) for v in range(n_od)
]
annotations_pc = [
    tuple(morse_graph_precomputed.annotations(v)) for v in range(n_pc)
]
assert annotations_od == annotations_pc, (
    f"Annotation mismatch:\n  on-demand: {annotations_od}\n"
    f"  precomputed: {annotations_pc}"
)

print(f"Both paths: {n_od} Morse nodes, {len(edges_od)} edges — outputs identical.")

Both paths: 4 Morse nodes, 3 edges — outputs identical.


## Summary

The table below summarises the wall-clock comparison.

In [8]:
print(f"{'Path':<28}  {'Wall-clock (s)':>16}")
print("-" * 46)
print(f"{'On-demand (corners)':<28}  {t_on_demand:>16.2f}")
print(f"{'Precomputed (total)':<28}  {t_precomputed_total:>16.2f}")
print(f"{'  precompute pass':<28}  {t_precompute:>16.2f}")
print(f"{'  CMGDB subdivision':<28}  {t_cmgdb_precomputed:>16.2f}")
ratio = t_precomputed_total / t_on_demand if t_on_demand > 0 else float('nan')
print(f"\nPrecomputed/on-demand ratio: {ratio:.2f}")
print(
    "(ratio > 1: on-demand is faster here because the analytic Leslie map is "
    "cheap per-call; precomputed overhead dominates.  For expensive maps such "
    "as neural networks the ratio inverts.)"
)
print("\nBoth paths produce identical Morse graphs (vertex count, edges, Conley indices).")

Path                            Wall-clock (s)
----------------------------------------------
On-demand (corners)                       6.98
Precomputed (total)                      43.59
  precompute pass                         1.90
  CMGDB subdivision                      41.69

Precomputed/on-demand ratio: 6.24
(ratio > 1: on-demand is faster here because the analytic Leslie map is cheap per-call; precomputed overhead dominates.  For expensive maps such as neural networks the ratio inverts.)

Both paths produce identical Morse graphs (vertex count, edges, Conley indices).
